# Bundestag API – Exploration
Direkte HTTP-Calls gegen `https://search.dip.bundestag.de/api/v1`

In [ ]:
import requests
import json
import pprint

BASE_URL = "https://search.dip.bundestag.de/api/v1"
API_KEY  = "I9FKdCn.hbfefNWCY336dL6x690GCU5PsGbDzlq0"  # öffentlicher Demo-Key

session = requests.Session()
session.headers.update({
    "Authorization": f"ApiKey {API_KEY}",
    "Accept": "application/json",
})

print("Session bereit.")

## Plenarprotokolle abrufen

In [ ]:
url = f"{BASE_URL}/plenarprotokoll"

params = {
    "wahlperiode": 20,
    "format": "json",
    "num": 5,
    "datum.start": "2024-01-01",
}

response = session.get(url, params=params, timeout=15)
print(f"Status : {response.status_code}")
print(f"URL    : {response.url}")

In [ ]:
data = response.json()

print(f"Anzahl Dokumente : {len(data.get('documents', []))}")
print(f"Cursor (next)    : {data.get('cursor', '–')}")
print()
print("Antwort-Keys:", list(data.keys()))

In [ ]:
# Erstes Dokument im Detail
erstes = data["documents"][0]
pprint.pprint(erstes)

In [ ]:
# Alle Dokumente tabellarisch
for dok in data["documents"]:
    print(f"{dok.get('datum', '?'):12}  id={dok.get('id', '?'):<8}  {dok.get('titel', '')[:80]}")

## Weitere Endpunkte

In [ ]:
def call_api(endpoint: str, **params) -> dict:
    """Hilfsfunktion: GET gegen einen beliebigen Endpunkt."""
    defaults = {"wahlperiode": 20, "format": "json", "num": 5}
    r = session.get(f"{BASE_URL}/{endpoint}", params={**defaults, **params}, timeout=15)
    r.raise_for_status()
    return r.json()

# Drucksachen
drucksachen = call_api("drucksache", **{"datum.start": "2024-01-01"})
print(f"Drucksachen: {len(drucksachen.get('documents', []))} gefunden")
for d in drucksachen["documents"]:
    print(f"  {d.get('datum', '?'):12}  {d.get('drucksachentyp', '?'):20}  {d.get('titel', '')[:60]}")

In [ ]:
# Personen / MdBs
personen = call_api("person", **{"fraktionMitgliedschaft.fraktion": "SPD"})
print(f"SPD-MdBs: {len(personen.get('documents', []))} gefunden")
for p in personen["documents"]:
    print(f"  {p.get('nachname', '?')}, {p.get('vorname', '?')}")